In [1]:
import re
import numpy
import json
import urllib3
from urllib.parse import urlparse, urlencode
from urllib.request import urlopen

In [185]:
base_page = 'https://www.techinterviewhandbook.org/behavioral-interview-questions/'

In [2]:
def get_html(base_url: str):
    parsed = urlparse(base_url)
    params = {'hl': 'en', 'safe': 'off'}
    encoded_params = urlencode(params)
    final_url = f"{parsed.scheme}://{parsed.netloc}{parsed.path}?{encoded_params}"
    http = urllib3.PoolManager()
    return http.request('GET', final_url)

In [186]:
html = get_html(base_page)

In [187]:
print(html)

In [188]:
body_part = html._body.decode("utf-8")
subs = re.findall(r'<li>[\w\s,.!:-?]*<\/li>', body_part)
res = []
for q in subs:
    splitted = re.split(r'>|<', q)
    res.append(splitted[2])

In [189]:
len(res)

94

In [84]:
with open("questions.json", "w") as file:
    json.dump(res, file)

In [174]:
base_awesome = 'https://github.com/ashishps1/awesome-behavioral-interviews'

In [190]:
html = get_html(base_awesome)
body_part = html._body.decode("utf-8")
subs = 'question'.join(re.split('question', re.split(r'article', body_part)[6])[15:20])
ss_splitted = re.split(r'summary', subs)

In [191]:
print(len(ss_splitted))

103


In [177]:
questions = []
answers = []
for i in range(1, 100, 2):
    questions.append(ss_splitted[i])
    answers.append(ss_splitted[i + 1])

In [178]:
questions = [q[19:-21].replace('\\n', ' ').strip() for q in questions]

In [179]:
new_answers = []
for answer in answers:
    a_splitted = re.split(r'\\u003[e|c/b|cb]', answer)
    ans = ''
    for s in a_splitted:
        s = re.sub(r'/lib|/lilib|/p|\\n|\n|/b', ' ', s)
        s = re.sub(r'[\s\n]+', ' ', s)
        if 'dir=' in s:
            continue
        if s == '/details' or s == 'details':
            continue
        if not s or len(s) < 10:
            continue
        ans += s.strip()
    new_answers.append(ans)

In [180]:
res_awesome = [{'question': q, 'answer': a} for q, a in zip(questions, new_answers)]

In [181]:
with open('../questions_awesome.json', 'w') as f:
    json.dump(res_awesome, f)

In [78]:
base_rubrics = 'https://www.techinterviewhandbook.org/behavioral-interview-rubrics/'

In [79]:
html = get_html(base_rubrics)
body_part = html._body.decode("utf-8")
body_splitted = re.split(r'article', body_part)
sub_questions = body_splitted[1].split('Example Questions')
sub_questions[6] = sub_questions[6].split('title')[0]

In [80]:
res_rubrics = []
ans = []
ques = []

for sub_q in sub_questions[1:6]:
    q, r = sub_q.split('Example Responses')
    q_split = q.split('</li><li>')
    ques.append(q_split[0][13:])
    for i in range(1, len(q_split) - 1):
        ques.append(q_split[i])
    ques.append(q_split[-1][:-13])

    r_split = re.split(r'<li>|</li>', r)
    for s in r_split:
        if s.startswith('Junior'):
            ans.append({'answer': s[8:], 'mark': 'bad'})
    
        if s.startswith('Senior'):
            ans.append({'answer': s[8:], 'mark': 'middle'})
    
        if s.startswith('Staff'):
            ans.append({'answer': s[7:], 'mark': 'good'})

    res_rubrics.append({'question': ques[0], 'answers': ans})
    for i in range(1, len(ques)):
        res_rubrics.append({'question': ques[i]})

    ans = []
    ques = []

In [81]:
print(res_rubrics)

[{'question': 'What project are you most proud of and why?', 'answers': [{'answer': 'A story about a project they are proud of that had an impact on their team.', 'mark': 'bad'}, {'answer': 'A story about a project they are proud of that had a large impact on their team.', 'mark': 'middle'}, {'answer': 'A story about a project they are proud of that had a large impact on their org.', 'mark': 'good'}]}, {'question': 'Tell me about a recent day working that was really great and/or fun.'}, {'question': 'Tell me about a time when you wanted to change something that was outside of your regular scope of work.', 'answers': [{'answer': 'A story about a change they proactively suggested and drove that had an impact on their team’s focus area. Usually only requiring the candidate themselves to work on.', 'mark': 'bad'}, {'answer': 'A story about a change they proactively suggested and drove that had an impact on their entire team. Usually requiring three or more people to work on.', 'mark': 'mid

In [82]:
with open('../questions_rubrics.json', 'w') as f:
    json.dump(res_rubrics, f)